In [1]:
!pip install -q av imageio decord opencv-python
!pip install -q sentence-transformers==2.6.1
!pip install -q -U bitsandbytes>=0.46.1
!pip install -q transformers==4.40.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 88.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.3/163.3 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 86.3 MB/s eta 0:00:00


In [2]:
# import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [3]:
"""
=============================================================================
AUTOPILOT VQA — InternVideo2.5 Pipeline (Sampled Frames Edition)
=============================================================================
Model  : OpenGVLab/InternVideo2_5_Chat_8B
Target : Kaggle T4 x2 (2 × 16 GB)

Reads pre-sampled JPEG frames from:
  {SAMPLED_FRAMES_DIR}/{video_id}/          ← VIDEO_SOURCE = "videos"
  {SAMPLED_FRAMES_DIR}/{video_id}_heatmap/  ← VIDEO_SOURCE = "heatmaps"

Video IDs are discovered from folder names (numeric folders only).
DATASET_PERCENT takes the first N% in sorted numeric order (0, 1, 2 ...).
Remaining IDs are written to unprocessed_video_ids.csv.

Two-pass approach:
  Pass 1 — Narrative : model sees all frames, describes the incident
  Pass 2 — Classify  : model sees same frames + narrative, answers JSON

Outputs:
  internvideo_raw_words.csv   — text answers
  internvideo_coded.csv       — integer codes
  unprocessed_video_ids.csv   — IDs deferred to next run

Install (once, then Session > Restart & Run All):
  !pip install -q av imageio opencv-python
  !pip install -q -U bitsandbytes>=0.46.1
  !pip install -q transformers==4.40.1
=============================================================================
"""

import os
import re
import gc
import json
import glob
import time
import warnings
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig

warnings.filterwarnings("ignore")


# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║                          CONFIGURATION                                   ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

# ── Model ──────────────────────────────────────────────────────────────────
MODEL_PATH = "OpenGVLab/InternVideo2_5_Chat_8B"

# ── Sampled frames directory ────────────────────────────────────────────────
# Folders inside: {video_id}/  and  {video_id}_heatmap/
SAMPLED_FRAMES_DIR = "/kaggle/input/datasets/shresthml/sample-frame/sampled_frames"

# ── Which folder suffix to use per video ────────────────────────────────────
# "videos"   → reads {SAMPLED_FRAMES_DIR}/{video_id}/
# "heatmaps" → reads {SAMPLED_FRAMES_DIR}/{video_id}_heatmap/
VIDEO_SOURCE = "videos"

# ── Output ─────────────────────────────────────────────────────────────────
OUTPUT_DIR            = "/kaggle/working"
SUBMISSION_WORDS_PATH = os.path.join(OUTPUT_DIR, "internvideo_raw_words.csv")
SUBMISSION_CODES_PATH = os.path.join(OUTPUT_DIR, "internvideo_coded.csv")
UNPROCESSED_CSV_PATH  = os.path.join(OUTPUT_DIR, "unprocessed_video_ids.csv")
SAMPLE_SUB_PATH       = "/kaggle/input/datasets/shresthml/vqa-samplesub/sample_submission.csv"
NARRATIVES_JSON_PATH = os.path.join(OUTPUT_DIR, "narratives.json")
# ── Dataset — always sorted numeric order, first N% processed ───────────────
DATASET_PERCENT = 100.0   # first 40% of video IDs by sorted numeric order

# ── Questions to ask in Pass 2 ──────────────────────────────────────────────
# Remove any key to skip it. Skipped keys get code -1 in output CSV.

QUESTIONS_TO_ASK = [
    "Q1a", "Q1b", "Q2"
]


# ── Quantization (NF4, T4 x2) ──────────────────────────────────────────────
# Do NOT pass torch_dtype / dtype to from_pretrained — custom __init__ rejects it.
# Model loads in fp16 by default; keep compute dtype fp16 to match.
QUANT_COMPUTE_DTYPE = torch.float16   # must match model default (fp16)
QUANT_TYPE          = "nf4"
QUANT_DOUBLE_QUANT  = True

# ── Frame preprocessing ─────────────────────────────────────────────────────
MAX_NUM_TILES    = 1      # 1 = no spatial tiling, saves VRAM
FRAME_INPUT_SIZE = 448    # InternVideo2.5 standard

# ── Generation ─────────────────────────────────────────────────────────────
MAX_NEW_TOKENS_NARRATIVE = 700
MAX_NEW_TOKENS_CLASSIFY  = 512
BASE_GENERATION_CONFIG = dict(
    do_sample=False,
    temperature=0.0,
    top_p=0.1,
    num_beams=1,
)

# ── Inference control ──────────────────────────────────────────────────────
MAX_RETRIES             = 2
UNMAPPABLE_CODE         = -2
UNKNOWN_CODE            = -1
SAVE_INTERMEDIATE_EVERY = 20
VERBOSE                 = True

# ── Debug ──────────────────────────────────────────────────────────────────
DEBUG_VIDEO_ID = None   # set to int video ID to test one video only


# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║                      QUESTION DEFINITIONS                                ║
# ║  ← PASTE YOUR OWN ALL_INCIDENT_QUESTIONS LIST HERE                      ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

ALL_INCIDENT_QUESTIONS = [
    # A) Time & Weather
    ("Q1a",
     "What was the time of day during the incident?",
     {"Day": 0, "Dawn (sunrise)": 1, "Dusk (sunset)": 2,
      "Night": 3, "Unknown": -1}),
    ("Q1b",
     "What were the weather conditions during the incident?",
     {"Sunny/Clear": 0, "Cloudy": 1, "Rainy": 2, "Partly Cloudy": 3,
      "Snowy": 4, "Foggy": 5, "Unknown(if sky not visible)": -1}),

    # B) Light Condition
    ("Q2",
     "What was the lighting condition at the time of the incident?",
     {"Daylight": 0, "Headlights only": 1,
      "Sunrise/Sunset – Early morning low-light, sun beginning to rise": 2,
      "Streetlights on": 4, "Unknown": -1}),

    # C) Traffic Environment
    ("Q3a",
     "Where did the incident take place (traffic environment)?",
     {"Urban area – City streets with dense buildings, intersections, and traffic": 0,
      "Suburban area – Residential or mixed-use commercial/residential area": 2,
      "Rural area – Open or undeveloped roadside with sparse traffic,no building or houses grasslands or feilds": 3,
      "Unknown": -1}),
    ("Q3b",
     "What facilities were present in the surrounding environment?",
     {"Urban area - skyscrapers, businesses, etc.": 1,
      "Construction zone / Work area – Road partially blocked or altered due to construction work": 2,
      "Residential Area (houses or living area not commercial or industrial zones,Presence of people walking,kids,Small streets or lanes)": 3,
      "Parking lot / Gas station – Off-road traffic environment with vehicles maneuvering": 4,
      "Near a school/college – Educational zone with potential pedestrian activity": 5,
      "Unknown": -1}),

    # D) Road Configuration
    ("Q4a",
     "What is the road configuration at the incident location?",
     {"Highway": 1, "Intersection": 2, "T-junction / Y-junction(3way road)": 3,
      "Residential street": 4, "Bridge / Overpass / Underpass": 5,
      "Roundabout / Traffic circle": 6, "Exit/Entrance ramp": 7,
      "Unknown": -1}),
    ("Q4b",
     "What is the road lane configuration?",
     {"Two-way divided(physical separator middle of road can be Concrete barrier,Metal railing,Green plants / curb median)": 0, "Two-way not divided(No physical divider)": 1,
      "One-way(All vehicles move in same direction)": 2, "Unknown": -1}),
    ("Q4c",
     "What is the lane direction of the Ego-Car at the time of the incident?",
     {"Right": 0, "Left": 1, "Middle": 2, "Unknown": -1}),

    # E) Incident Type
    ("Q5a",
     "What is the category of the incident?",
     {"Pedestrian/Cyclist/Vulnerable Road Users": 0,
      "Vehicle-to-Vehicle(incident involving 2 vehicle)": 1,
      "Animals / Objects(incident related to animals)": 2,
      "No Collision(normal driving scene no incident of animal near miss,no loss of control or collision)": 3,
      "Barriers / Fixed Objects(collision with barriers or fixed object)": 5,
      "Vehicle Loss of Control / Rollover(vechicle spins,skid,Swerving,or vehicle tips or flips onto its side or roof)": 6,
      "Unknown(unsure)": -1}),
    ("Q5b",
     "Primary Incident Entity — who caused or could have prevented the incident?",
     {"Pedestrian": 0, "Ego-car": 1, "Animal": 2, "No collision(when everything normal)": 3,
      "Another Vehicle": 5, "Cyclist": 6, "Flying Object": 10,
      "Smoke": 13, "Object on the road": 29, "Unknown": -1}),
    ("Q5c",
     "If the Primary Incident Entity(who caused incident) is a vehicle, what type of vehicle is it?",
     {"Small Car": 0, "No collision": 1, "Truck": 2, "Bicycle": 3,
      "Bicycle Cart": 4, "Motorcycle": 5, "Scooter": 6,
      "SUV": 8, "Bus": 9, "Not applicable(primary entity not a vehicle)": -2, "Unknown(no incident everything normal so no primary entity)": -1}),
    ("Q5e",
     "What was the Primary Entity(who caused incident) doing at the time of the incident?",
     {"Crossing": 0, "Speeding": 1, "Driving normally": 3,
      "Overtaking/Changing Lanes": 4, "Turning": 6, "Walking": 7,
      "Stopped": 8, "Swerving(sudden, sharp movement of a vehicle sideways (left/right)) ": 10, "Ignoring traffic signal": 11,
      "Lost control": 12, "Fell Down": 13, "Rolling through(vehicle slows down but does NOT fully stop when it should)": 14,
      "Failure to yield": 15, "Flying(flying object or debris)": 16, "Rolling over": 17,
      "Stationary": 18, "Fallen into the road": 19, "Unknown(no incident everything normal so no primary entity)": -1}),
    ("Q5f",
     "Secondary Incident Entity — the victim or 2nd entity involved in incident or entity that was hit:",
     {"Cyclist": 0, "Ego-car": 1, "Another Vehicle": 2,
      "Object or Barrier": 3, "Pedestrian": 4, "Scooter": 5,
      "Animal": 6, "Unknown(no incident)": -1}),
    ("Q5g",
     "If the Secondary Incident Entity(the victim or 2nd entity involved in incident or entity that was hit) is a vehicle, what type of vehicle is it?",
     {"Scooter": 0, "Small Car": 1, "Bicycle": 2, "Truck": 3,
      "SUV": 4, "Bus": 5, "Not applicable(if not vehicle)": -2, "Unknown": -1}),
    ("Q5i",
     "What was the Secondary Entity(the victim or 2nd entity involved in incident or entity that was hit) doing at the time of the incident?",
     {"Driving normally": 0, "Swerving": 1, "Fixed Position": 2,
      "Speeding": 3, "Crossing": 4, "Turning": 5, "Lost control": 6,
      "Failure to yield": 7, "Flying": 8, "Stopped": 9,
      "Fallen onto road": 10, "Overtaking/Changing Lanes": 11,
      "Rolling over": 13, "UTurn": 14, "Unknown": -1}),
    ("Q5j",
     "What was the peak outcome of the incident?",
     {"Near-miss": 0, "No collision": 1, "Collision": 2,
      "No effect": 3, "Avoided in advance": 4, "Chain-reaction": 5,
      "Unknown": -1}),

    # F) Incident Prevention Measure
    ("Q6a",
     "How could this incident most likely have been prevented by the Primary Entity?",
     {"Primary Entity was traveling unsafe": 0,
      "No Fault: Primary Entity was behaving normally but could not prevent incident": 1,
      "Primary Entity could have maneuvered but did not": 2,
      "No Incident": 3,
      "No Fault: Primary Entity avoided Incident": 4,
      "Primary Entity is not capable of preventing accident (animal/inanimate)": 5,
      "Primary entity had defect (object/part fell off, equipment failure)": 6,
      "Unknown": -1}),
    ("Q6b",
     "How could this incident most likely have been prevented by the Secondary Entity?",
     {"No Fault: Secondary Entity avoided Incident": 0,
      "No Fault: Secondary Entity was behaving normally but could not prevent accident": 1,
      "Secondary Entity is not capable of preventing accident (animal/inanimate)": 2,
      "Secondary Entity was traveling unsafe": 3,
      "No Incident": 4,
      "Secondary entity had defect (object/part fell off, equipment failure)": 5,
      "Secondary Entity could have maneuvered but did not": 6,
      "Unknown": -1}),

    # G) Initial Point of Impact
    ("Q7a",
     "If the Primary Entity was a vehicle, where on the vehicle did the initial impact occur?",
     {"No collision": 0, "Front": 1, "Left side": 2, "Right side": 4,
      "Rear": 5, "Top/Roof": 6, "Unknown": -1}),
    ("Q7b",
     "If the Primary Entity was a vehicle, what side of the impacted area was hit?",
     {"No collision": 0, "Left Corner": 2, "Front": 3,
      "Right Side of Vehicle": 4, "Right Corner": 5,
      "Undercarriage / Bottom": 6, "Left Side of Vehicle": 7,
      "Unknown": -1}),
    ("Q7c",
     "If the Secondary Entity was a vehicle, where on the vehicle did the initial impact occur?",
     {"No Collision": 0, "Right side": 1, "Front": 2, "Top/Roof": 3,
      "Left side": 5, "Rear": 6, "Unknown": -1}),
    ("Q7d",
     "If the Secondary Entity was a vehicle, what side of the impacted area was hit?",
     {"No collision": 0, "Left Corner": 1, "Right Corner": 3,
      "Left Side of the Vehicle": 4, "Roof/Top": 5,
      "Right Side of Vehicle": 6, "Front Grill/Bumper": 7,
      "Mid-Right Side": 9, "Mid-Left Side of Truck": 10,
      "Undercarriage / Bottom": 11, "Rear": 12, "Unknown": -1}),

    # H) Traffic Control & Signage
    ("Q8",
     "What traffic control or road sign is present at the incident location?",
     {"Yield / Give way sign": 0, "None visible": 1, "Traffic light": 3,
      "Construction / Work zone sign": 4, "No entry / One-way sign": 5,
      "Pedestrian crossing": 6, "Traffic light Ahead Sign": 7,
      "School zone / Warning sign": 8, "Speed limit sign": 9,
      "Stop sign": 10, "Railroad crossing sign / gate": 11,
      "Unknown": -1}),

    # I) Road Surface & Material
    ("Q9a",
     "What was the road surface condition during the incident?",
     {"Dry pavement": 0, "Wet road": 1, "Snow / Ice covered": 2,
      "Debris / Mud on road": 3, "Unknown": -1}),
    ("Q9b",
     "What type of road material is the vehicle driving on?",
     {"Asphalt / Blacktop": 0, "Concrete": 1, "Gravel / Dirt": 3,
      "Unknown": -1}),
]




ALL_Q_KEYS    = [q[0] for q in ALL_INCIDENT_QUESTIONS]
LABEL_TO_CODE = {k: {lab.lower().strip(): code for lab, code in codes.items()}
                 for k, _, codes in ALL_INCIDENT_QUESTIONS}
VALID_CODES   = {k: set(codes.values()) | {-2}
                 for k, _, codes in ALL_INCIDENT_QUESTIONS}

# Primary entity codes that are NOT vehicles → Q5c must be -2
NON_VEHICLE_PRIMARY   = {0, 2, 3, 6, 10, 13, 29}
# Secondary entity codes that are NOT vehicles → Q5g must be -2
NON_VEHICLE_SECONDARY = {0, 3, 4, 5, 6}


# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║              CSV COLUMN HEADERS (match sample_submission.csv)            ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

# Full KEY → exact CSV column header mapping for all questions
KEY_TO_COLUMN = {
    # A) Weather
    "Q1a": "A)Weather:\nQ1.a: What was the time of day during the incident?",
    "Q1b": "A)Weather:\nQ1.b: What were the weather conditions during the incident?",
    # B) Light
    "Q2":  "B) Light Condition:\nQ2: What was the lighting condition at the time of the incident?",
    # C) Traffic Environment
    "Q3a": "C) Traffic Environment:\nQ3.a: Where did the incident take place (traffic environment)?",
    "Q3b": "C) Traffic Environment:\nQ3.b: What facilities were present in the surrounding environment?",
    # D) Road Configuration
    "Q4a": "D) Road Configuration\nQ4.a: What is the road configuration at the incident time?",
    "Q4b": "D) Road Configuration\nQ4.b: Road lane configuration:",
    "Q4c": "D) Road Configuration\nQ4.c: Lane direction of Ego-Car at time of incident:",
    # E) Incident Type
    "Q5a": "E) Incident Type\nQ5.a: What is the category of the incident?",
    "Q5b": "E) Incident Type\nQ5.b: Primary Incident Entity (Who caused or could have prevented the incident)?",
    "Q5c": "E) Incident Type\nQ5.c: If Primary Incident Entity is a vehicle, what type of vehicle is it?",
    "Q5e": "E) Incident Type\nQ5.e: Primary Entity Behavior:",
    "Q5f": "E) Incident Type\nQ5.f Secondary Incident Entity (Victim, secondary cause or entity that was hit):",
    "Q5g": "E) Incident Type:\nQ5.g: If Secondary Incident Entity is a vehicle, what type of vehicle is it?",
    "Q5i": "E) Incident Type\nQ5.i: Secondary Entity Behavior:",
    "Q5j": "E) Incident Type\nQ5.j: Incident peak",
    # F) Prevention
    "Q6a": "F) Incident Prevention Measure\nQ6.a: How could this Incident most likely have been prevented by the Primary Entity?",
    "Q6b": "F) Incident Prevention Measure\nQ6.b: How could this incident most likely have been prevented by the Secondary Entity?",
    # G) Impact
    "Q7a": "G)Initial Point of Impact\nQ7.a: If the Primary Entity was a vehicle, where on the vehicle did the initial impact occur?",
    "Q7b": "G)Initial Point of Impact\nQ7.b: If the Primary Entity was a vehicle, what side of the impacted area was hit?",
    "Q7c": "G)Initial Point of Impact\nQ7.c: If the Secondary Entity was a vehicle, where on the vehicle did the initial impact occur?",
    "Q7d": "G)Initial Point of Impact\nQ7.d: If the Secondary Entity was a vehicle, what side of the impacted area was hit?",
    # H) Signage
    "Q8":  "H) Traffic Control & Signage Presence:\nQ8: What traffic control or road sign is present at the incident location?",
    # I) Road Surface
    "Q9a": "I) Road Surface & Material Condition\nQ9.a: What was the road surface condition during the incident?",
    "Q9b": "I) Road Surface & Material Condition\nQ9.b: What type of road material is the vehicle driving on?",
}


# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║                      WORD → CODE MAPPING                                 ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

WORD_MAPPINGS = {
    # ── A) Weather ────────────────────────────────────────────────────────────
    "Q1a": [
        (["daytime", "day time", "afternoon", "midday", "noon",
          "broad daylight", "during the day", "morning"], 0),
        (["dawn", "sunrise", "early morning", "first light",
          "break of day", "just after sunrise"], 1),
        (["dusk", "sunset", "twilight", "evening", "late evening",
          "golden hour", "just before dark"], 2),
        (["night", "nighttime", "night time", "dark", "darkness",
          "late night", "midnight", "after dark"], 3),
        (["unknown", "unclear", "not sure", "cannot tell",
          "unsure", "not visible", "indeterminate"], -1),
    ],
    "Q1b": [
        (["sunny", "clear", "clear sky", "bright", "fair weather",
          "no clouds", "cloudless", "sunshine"], 0),
        (["cloudy", "overcast", "grey sky", "gray sky",
          "heavy clouds", "dark clouds", "full cloud cover"], 1),
        (["rainy", "raining", "rain", "wet", "drizzle", "drizzling",
          "downpour", "shower", "light rain", "heavy rain"], 2),
        (["partly cloudy", "partial clouds", "some clouds",
          "scattered clouds", "mostly clear", "intermittent clouds"], 3),
        (["snowy", "snow", "snowing", "snowfall", "blizzard",
          "sleet", "icy", "hail"], 4),
        (["foggy", "fog", "mist", "misty", "hazy", "haze",
          "low visibility", "thick fog"], 5),
        (["unknown", "unclear", "not sure", "cannot determine",
          "weather not visible"], -1),
    ],

    # ── B) Light Condition ────────────────────────────────────────────────────
    "Q2": [
        (["daylight", "daytime lighting", "natural light",
          "well lit by sun", "bright daylight", "full daylight"], 0),
        (["headlights", "headlights only", "using headlights",
          "vehicle lights", "car lights on", "no streetlights"], 1),
        (["sunrise", "sunset", "low light", "early morning light",
          "dusk lighting", "dawn lighting", "low sun", "golden hour light"], 2),
        (["streetlights", "street lights", "street lamps", "lamp posts",
          "artificial lighting", "road lights", "lit by streetlights"], 4),
        (["unknown", "unclear", "unsure", "cannot tell",
          "lighting not visible"], -1),
    ],

    # ── C) Traffic Environment ────────────────────────────────────────────────
    "Q3a": [
        (["urban", "city", "downtown", "city street", "dense traffic",
          "metropolitan", "town center", "commercial district",
          "busy street", "intersection-heavy"], 0),
        (["suburban", "residential", "mixed-use", "suburb",
          "outskirts", "housing area", "neighborhood street"], 2),
        (["rural", "countryside", "open road", "highway outskirts",
          "remote", "undeveloped", "sparse traffic", "country road",
          "farmland", "outback"], 3),
        (["unknown", "unclear", "cannot determine",
          "unsure of location"], -1),
    ],
    "Q3b": [
        (["skyscraper", "business district", "commercial area",
          "office buildings", "shops", "urban buildings",
          "high-rise", "city buildings"], 1),
        (["construction", "work zone", "road work", "construction zone",
          "workers on road", "lane closure", "roadwork"], 2),
        (["residential", "houses", "homes", "apartments",
          "neighborhood", "housing", "suburban homes"], 3),
        (["parking lot", "gas station", "petrol station", "parking area",
          "service station", "car park", "fuel station"], 4),
        (["school", "college", "university", "school zone",
          "campus", "educational zone", "school area"], 5),
        (["unknown", "unclear", "no facility visible",
          "cannot determine", "not visible"], -1),
    ],

    # ── D) Road Configuration ─────────────────────────────────────────────────
    "Q4a": [
        (["highway", "motorway", "freeway", "expressway",
          "interstate", "dual carriageway", "fast road"], 1),
        (["intersection", "crossroads", "four-way", "4-way",
          "cross junction", "crossing junction"], 2),
        (["t-junction", "t junction", "y-junction", "y junction",
          "three-way junction", "forked road", "fork in road"], 3),
        (["residential street", "local street", "neighborhood road",
          "side street", "quiet road", "housing street"], 4),
        (["bridge", "overpass", "underpass", "flyover",
          "viaduct", "tunnel", "elevated road"], 5),
        (["roundabout", "traffic circle", "rotary", "circular junction",
          "ring junction", "carousel junction"], 6),
        (["exit ramp", "entrance ramp", "on-ramp", "off-ramp",
          "highway ramp", "merge lane", "slip road"], 7),
        (["unknown", "unclear", "cannot determine",
          "unsure of road type"], -1),
    ],
    "Q4b": [
        (["two-way divided", "divided road", "median divided",
          "dual carriageway", "divided highway", "center divider"], 0),
        (["two-way undivided", "two way not divided", "undivided road",
          "no median", "no divider", "two-way street", "bidirectional"], 1),
        (["one-way", "one way", "single direction", "one direction traffic",
          "one-way street", "unidirectional"], 2),
        (["unknown", "unclear", "cannot determine", "lane type unclear"], -1),
    ],
    "Q4c": [
        (["right lane", "right side", "rightmost lane",
          "far right", "driving on right"], 0),
        (["left lane", "left side", "leftmost lane",
          "far left", "driving on left"], 1),
        (["middle lane", "center lane", "central lane",
          "mid lane", "in the middle"], 2),
        (["unknown", "unclear", "cannot determine",
          "unsure of lane position"], -1),
    ],

    # ── E) Incident Type ──────────────────────────────────────────────────────
    "Q5a": [
        (["vehicle-to-vehicle", "vehicle to vehicle", "car crash", "v2v",
          "car collision", "multi-vehicle", "two vehicles collide",
          "vehicle collision", "cars collide"], 1),
        (["loss of control", "rollover", "lost control", "flip",
          "overturn", "rolls over", "vehicle flipped", "car flipped",
          "skidded off", "spun out"], 6),
        (["barrier", "fixed object", "guardrail", "wall", "pole",
          "fence", "median", "divider", "railing"], 5),
        (["no collision", "no crash", "no contact", "no incident",
          "nothing happened", "uneventful"], 3),
        (["pedestrian", "cyclist", "vulnerable", "walker",
          "biker", "bicyclist", "person hit", "hit a person",
          "person was struck"], 0),
        (["animal", "object", "debris", "obstacle", "fallen tree",
          "snake", "dog", "cat", "cow", "deer", "bird", "rat",
          "fallen object on road"], 2),
    ],
    "Q5b": [
        (["ego-car", "ego car", "ego vehicle", "own car",
          "dashcam car", "camera car", "dashcam vehicle",
          "filming vehicle", "our vehicle", "host vehicle"], 1),
        (["no collision", "no crash", "no incident"], 3),
        (["flying object", "projectile", "airborne object",
          "flying debris", "airborne debris", "object flying",
          "board flying", "box flying", "flying board"], 10),
        (["smoke", "fume", "exhaust", "thick smoke"], 13),
        (["object on the road", "road object", "debris on road",
          "fallen object", "obstacle on road", "object lying on road",
          "landslide", "fallen tree on road", "rock on road"], 29),
        (["pedestrian", "person", "walker", "jaywalker",
          "individual crossing", "person crossing"], 0),
        (["animal", "dog", "cat", "cow", "deer", "bear",
          "snake", "bird", "rat", "goat", "sheep", "horse", "monkey"], 2),
        (["cyclist", "bicycle rider", "bicyclist", "bike rider",
          "person on bicycle", "person on bike"], 6),
        (["another vehicle", "other vehicle", "truck", "bus",
          "van", "car", "vehicle", "scooter", "motorcycle",
          "motorbike", "suv", "sedan"], 5),
    ],
    "Q5c": [
        (["not applicable", "n/a", "not a vehicle",
          "no vehicle", "does not apply"], -2),
        (["no collision"], 1),
        (["bicycle cart", "cargo bike", "rickshaw",
          "cycle rickshaw", "tricycle cargo"], 4),
        (["motorcycle", "motorbike", "moped", "two-wheeler"], 5),
        (["scooter", "moped scooter"], 6),
        (["bicycle", "bike", "pushbike", "cycle", "person on cycle"], 3),
        (["truck", "lorry", "semi", "pickup", "van",
          "freight", "cargo", "delivery", "tanker",
          "trailer", "heavy vehicle"], 2),
        (["suv", "crossover", "4x4", "jeep", "sport utility vehicle"], 8),
        (["bus", "coach", "minibus", "transit bus", "autobus", "city bus"], 9),
        (["small car", "sedan", "hatchback", "car", "vehicle",
          "automobile", "coupe", "compact car", "passenger car"], 0),
    ],
    "Q5e": [
        (["fallen into the road", "fallen into road", "fell into road",
          "collapsed into road", "fell onto roadway"], 19),
        (["rolling over", "rolled over", "overturned",
          "flipped over", "vehicle overturned", "car rolled over"], 17),
        (["rolling through", "rolled through",
          "roll through", "rolling past stop"], 14),
        (["fell down", "fallen down", "tripped", "stumbled",
          "fell", "person fell", "fell from vehicle"], 13),
        (["ignoring traffic signal", "ran red light", "disobeyed signal",
          "ran signal", "ignored light", "ran through red"], 11),
        (["failure to yield", "failed to yield",
          "no yield", "did not yield", "did not give way"], 15),
        (["overtaking", "changing lanes", "lane change", "cutting in",
          "merging", "switching lanes", "cutting across lanes"], 4),
        (["lost control", "loss of control", "skidded",
          "out of control", "lost traction", "spun out of control"], 12),
        (["driving normally", "normal driving", "cruising",
          "proceeding normally", "driving as usual", "moving normally"], 3),
        (["stationary", "standing still", "not moving",
          "idle", "parked", "completely stopped and stationary"], 18),
        (["stopped", "halted", "braked", "came to stop",
          "pulled over", "applied brakes", "stopped suddenly"], 8),
        (["swerving", "swerved", "veering", "veer", "weaving", "weaved"], 10),
        (["speeding", "fast", "high speed", "accelerating",
          "excessive speed", "at high speed", "travelling fast"], 1),
        (["crossing", "jaywalking", "crossed", "crossing road",
          "crossing street", "walking across road"], 0),
        (["turning", "turn", "making a turn", "turned", "taking a turn"], 6),
        (["walking", "walk", "on foot", "strolling", "walking along"], 7),
        (["flying", "airborne", "launched", "became airborne", "was flying"], 16),
    ],
    "Q5f": [
        (["ego-car", "ego car", "own vehicle", "dashcam car",
          "camera vehicle", "dashcam vehicle", "filming vehicle",
          "the ego vehicle", "our car"], 1),
        (["object or barrier", "barrier", "guardrail", "wall", "pole",
          "fence", "fixed object", "railing", "median", "divider",
          "stationary object", "tree", "fallen tree"], 3),
        (["scooter"], 5),
        (["animal", "dog", "cat", "cow", "deer", "horse", "snake",
          "goat", "sheep", "bird", "rat", "monkey"], 6),
        (["cyclist", "bicycle rider", "bicyclist",
          "bike rider", "person on bicycle"], 0),
        (["pedestrian", "person", "walker", "individual",
          "person on foot", "person walking"], 4),
        (["another vehicle", "other vehicle", "vehicle", "car", "truck",
          "bus", "motorcycle", "motorbike", "van", "suv", "sedan"], 2),
    ],
    "Q5g": [
        (["not applicable", "n/a", "not a vehicle",
          "no vehicle", "does not apply"], -2),
        (["scooter"], 0),
        (["bicycle", "bike", "pushbike", "cycle"], 2),
        (["truck", "lorry", "semi", "pickup",
          "van", "heavy vehicle", "cargo vehicle"], 3),
        (["suv", "crossover", "4x4", "jeep"], 4),
        (["bus", "coach", "minibus", "transit"], 5),
        (["small car", "sedan", "hatchback", "car", "vehicle",
          "automobile", "passenger car", "compact"], 1),
    ],
    "Q5i": [
        (["fallen onto road", "fallen on road", "fell onto road",
          "collapsed on road", "lying on road", "person lying on road"], 10),
        (["rolling over", "flipped", "overturned",
          "vehicle rolled over", "car flipped"], 13),
        (["overtaking", "changing lanes", "lane change", "switching lanes"], 11),
        (["failure to yield", "failed to yield", "did not yield", "no yield"], 7),
        (["lost control", "skidded", "out of control", "loss of control"], 6),
        (["fixed position", "parked", "stationary", "standing still",
          "fixed in place", "not moving at all", "static"], 2),
        (["driving normally", "normal driving", "cruising",
          "moving normally", "driving as usual"], 0),
        (["swerving", "swerved", "veering", "weaving", "weaved"], 1),
        (["speeding", "fast", "high speed", "moving fast", "excessive speed"], 3),
        (["crossing", "jaywalking", "crossing road", "crossing street"], 4),
        (["turning", "turn", "making a turn", "turned"], 5),
        (["flying", "airborne", "was flying", "in the air"], 8),
        (["stopped", "halted", "braking", "came to stop",
          "braked hard", "emergency stop"], 9),
        (["u-turn", "uturn", "turning around", "reversing direction"], 14),
    ],
    "Q5j": [
        (["near-miss", "near miss", "close call", "almost hit",
          "narrowly avoided", "barely missed", "barely avoided",
          "just missed", "just avoided"], 0),
        (["chain-reaction", "chain reaction", "pile-up", "multi-car",
          "secondary collision", "caused another crash"], 5),
        (["avoided in advance", "avoided early", "preemptive",
          "proactively avoided", "avoided beforehand",
          "stopped well in advance"], 4),
        (["no effect", "no impact", "uneventful", "nothing happened",
          "no consequence", "passed by safely"], 3),
        (["no collision", "no crash", "no contact", "no impact",
          "no accident", "did not collide", "did not hit"], 1),
        (["collision", "crash", "impact", "hit", "struck", "collided",
          "contact made", "made contact", "bumped into", "ran into",
          "slammed into", "rear-ended", "side-swiped"], 2),
    ],

    # ── F) Incident Prevention ────────────────────────────────────────────────
    "Q6a": [
        (["unsafe speed", "traveling unsafe", "too fast", "speeding",
          "reckless", "unsafe driving", "driving dangerously"], 0),
        (["no fault", "behaving normally", "could not prevent",
          "not at fault", "unavoidable", "acting normally"], 1),
        (["could have maneuvered", "failed to maneuver", "did not swerve",
          "did not steer away", "could have avoided",
          "did not take evasive action"], 2),
        (["no incident", "nothing happened", "no event",
          "uneventful", "no collision occurred"], 3),
        (["avoided", "primary entity avoided", "evaded",
          "successfully avoided", "took evasive action"], 4),
        (["animal", "inanimate", "not capable", "cannot prevent",
          "incapable of preventing", "object fell"], 5),
        (["defect", "equipment failure", "part fell off",
          "mechanical failure", "brake failure", "tire blowout"], 6),
        (["unknown", "unclear", "cannot determine"], -1),
    ],
    "Q6b": [
        (["secondary avoided", "secondary entity avoided",
          "evaded successfully", "no fault avoided"], 0),
        (["no fault", "behaving normally", "could not prevent",
          "not at fault", "unavoidable", "acting normally"], 1),
        (["animal", "inanimate", "not capable", "cannot prevent",
          "incapable of preventing"], 2),
        (["unsafe", "traveling unsafe", "secondary unsafe",
          "secondary speeding", "reckless secondary"], 3),
        (["no incident", "nothing happened", "no event",
          "uneventful", "no collision occurred"], 4),
        (["defect", "equipment failure", "part fell off",
          "mechanical failure", "brake failure", "tire blowout"], 5),
        (["could have maneuvered", "failed to maneuver",
          "did not swerve", "could have avoided",
          "did not take evasive action", "secondary did not avoid"], 6),
        (["unknown", "unclear", "cannot determine"], -1),
    ],

    # ── G) Initial Point of Impact ────────────────────────────────────────────
    "Q7a": [
        (["no collision", "no impact", "no contact",
          "did not collide", "no crash"], 0),
        (["front", "front end", "front bumper", "hood",
          "frontal impact", "head-on"], 1),
        (["left side", "driver side", "left flank",
          "left door", "left panel"], 2),
        (["right side", "passenger side", "right flank",
          "right door", "right panel"], 4),
        (["rear", "back", "rear end", "rear bumper",
          "tail", "rear-ended", "back of vehicle"], 5),
        (["top", "roof", "rooftop", "top of vehicle",
          "roof impact", "overhead"], 6),
        (["unknown", "unclear", "cannot determine",
          "impact point unknown"], -1),
    ],
    "Q7b": [
        (["no collision", "no impact", "no contact"], 0),
        (["left corner", "front-left corner", "left front corner",
          "driver-side corner", "left-front"], 2),
        (["front", "front center", "straight front",
          "front grill", "frontal center"], 3),
        (["right side", "right flank", "passenger side",
          "right door area", "right panel"], 4),
        (["right corner", "front-right corner", "right front corner",
          "passenger-side corner", "right-front"], 5),
        (["undercarriage", "bottom", "underbody",
          "underneath", "floor"], 6),
        (["left side", "left flank", "driver side area",
          "left panel area", "left door area"], 7),
        (["unknown", "unclear", "cannot determine"], -1),
    ],
    "Q7c": [
        (["no collision", "no impact", "no contact",
          "did not collide", "no crash"], 0),
        (["right side", "passenger side", "right flank",
          "right panel", "right door"], 1),
        (["front", "front end", "front bumper",
          "frontal impact", "head-on front"], 2),
        (["top", "roof", "rooftop", "top of vehicle",
          "roof hit", "overhead impact"], 3),
        (["left side", "driver side", "left flank",
          "left panel", "left door"], 5),
        (["rear", "back", "rear end", "rear bumper",
          "tail end", "back of vehicle"], 6),
        (["unknown", "unclear", "cannot determine"], -1),
    ],
    "Q7d": [
        (["no collision", "no impact", "no contact"], 0),
        (["left corner", "front-left", "driver front corner",
          "left-front corner"], 1),
        (["right corner", "front-right", "passenger front corner",
          "right-front corner"], 3),
        (["left side", "driver side", "left flank",
          "left door panel"], 4),
        (["roof", "top", "rooftop", "overhead"], 5),
        (["right side", "passenger side", "right flank",
          "right panel"], 6),
        (["front grill", "grille", "front bumper", "front center",
          "bumper"], 7),
        (["mid-right", "middle right side", "center right"], 9),
        (["mid-left", "middle left side", "center left"], 10),
        (["undercarriage", "bottom", "underbody", "underneath"], 11),
        (["rear", "back", "tail", "rear bumper", "rear end"], 12),
        (["unknown", "unclear", "cannot determine"], -1),
    ],

    # ── H) Traffic Control & Signage ──────────────────────────────────────────
    "Q8": [
        (["yield", "give way", "yield sign", "give way sign",
          "yield marking"], 0),
        (["none", "no sign", "no signal", "no traffic control",
          "no signage visible", "no markings"], 1),
        (["traffic light", "traffic signal", "red light",
          "green light", "signal light", "stoplight"], 3),
        (["construction sign", "work zone sign", "roadwork sign",
          "orange sign", "construction warning"], 4),
        (["no entry", "one-way sign", "do not enter",
          "wrong way sign", "one way"], 5),
        (["pedestrian crossing", "crosswalk", "zebra crossing",
          "pedestrian sign", "crossing sign"], 6),
        (["traffic light ahead", "signal ahead sign",
          "traffic light warning sign"], 7),
        (["school zone", "school sign", "school warning",
          "children crossing", "school area sign"], 8),
        (["speed limit", "speed sign", "speed limit sign",
          "max speed", "speed restriction"], 9),
        (["stop sign", "full stop", "octagonal sign"], 10),
        (["railroad crossing", "level crossing", "train crossing",
          "railway gate", "railroad gate"], 11),
        (["unknown", "unclear", "cannot determine",
          "signage not visible"], -1),
    ],

    # ── I) Road Surface & Material ────────────────────────────────────────────
    "Q9a": [
        (["dry", "dry road", "dry pavement", "dry surface",
          "no moisture", "clear road surface"], 0),
        (["wet", "wet road", "wet surface", "after rain",
          "damp road", "slippery wet", "puddles"], 1),
        (["snow", "ice", "snowy road", "icy road", "ice covered",
          "snow covered", "frozen road", "black ice"], 2),
        (["debris", "mud", "muddy", "gravel", "dirt on road",
          "rocks on road", "rubble", "sand on road"], 3),
        (["unknown", "unclear", "cannot determine",
          "road condition unknown"], -1),
    ],
    "Q9b": [
        (["asphalt", "blacktop", "tarmac", "bitumen",
          "paved road", "sealed road", "asphalt road"], 0),
        (["concrete", "cement", "concrete road", "concrete pavement",
          "cement road", "concrete slab"], 1),
        (["gravel", "dirt", "unpaved", "gravel road", "dirt road",
          "dirt track", "unsealed road", "stone road"], 3),
        (["unknown", "unclear", "cannot determine",
          "road material unclear"], -1),
    ],
}


# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║                      WORD → CODE CONVERTER                               ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def word_to_code(raw: str, q_key: str) -> int:
    if not raw or not isinstance(raw, str):
        return UNMAPPABLE_CODE

    cleaned = re.sub(r"\s+", " ", raw.lower().strip().strip("\"'.,;:"))

    if cleaned in ("unknown", "uncertain", "unclear", "not sure", "n/a", "na"):
        return UNKNOWN_CODE

    label_map = LABEL_TO_CODE.get(q_key, {})
    if cleaned in label_map:
        return label_map[cleaned]
    stripped = cleaned.rstrip(".,;:")
    if stripped in label_map:
        return label_map[stripped]

    for keywords, code in WORD_MAPPINGS.get(q_key, []):
        for kw in keywords:
            if kw in cleaned:
                return code

    return UNMAPPABLE_CODE



# Model

In [4]:

# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║              INTERNVIDEO2.5 FRAME UTILITIES                              ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)


def build_transform(input_size: int = FRAME_INPUT_SIZE):
    return T.Compose([
        T.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float("inf")
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        ratio_diff = abs(aspect_ratio - ratio[0] / ratio[1])
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio


def dynamic_preprocess(image, min_num=1, max_num=MAX_NUM_TILES,
                       image_size=FRAME_INPUT_SIZE, use_thumbnail=False):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height

    target_ratios = set(
        (i, j)
        for n in range(min_num, max_num + 1)
        for i in range(1, n + 1)
        for j in range(1, n + 1)
        if min_num <= i * j <= max_num
    )
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])
    target_ar = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size
    )

    tw     = image_size * target_ar[0]
    th     = image_size * target_ar[1]
    blocks = target_ar[0] * target_ar[1]
    resized = image.resize((tw, th))

    tiles = []
    for i in range(blocks):
        box = (
            (i  % (tw // image_size)) * image_size,
            (i  // (tw // image_size)) * image_size,
            ((i % (tw // image_size)) + 1) * image_size,
            ((i // (tw // image_size)) + 1) * image_size,
        )
        tiles.append(resized.crop(box))

    assert len(tiles) == blocks
    if use_thumbnail and len(tiles) != 1:
        tiles.append(image.resize((image_size, image_size)))
    return tiles


def frames_dir_for(video_id: int) -> str:
    """Return the correct frames folder path for a given video_id."""
    if VIDEO_SOURCE == "heatmaps":
        return os.path.join(SAMPLED_FRAMES_DIR, f"{video_id}_heatmap")
    return os.path.join(SAMPLED_FRAMES_DIR, str(video_id))


def load_sampled_frames(frames_dir: str,
                        input_size: int = FRAME_INPUT_SIZE,
                        max_num: int = MAX_NUM_TILES):
    """
    Load all JPEG frames from frames_dir (sorted by filename).
    Returns (pixel_values, num_patches_list) ready for model.chat().
    pixel_values     : Float32 CPU Tensor [sum_patches, 3, H, W]
    num_patches_list : list[int], one entry per frame
    """
    jpeg_files = sorted(glob.glob(os.path.join(frames_dir, "*.jpg")))
    if not jpeg_files:
        jpeg_files = sorted(glob.glob(os.path.join(frames_dir, "*.jpeg")))
    if not jpeg_files:
        raise FileNotFoundError(f"No JPEG frames found in: {frames_dir}")

    transform = build_transform(input_size)
    pv_list   = []
    np_list   = []

    for fp in jpeg_files:
        img   = Image.open(fp).convert("RGB")
        tiles = dynamic_preprocess(img, image_size=input_size,
                                   use_thumbnail=True, max_num=max_num)
        pv    = torch.stack([transform(t) for t in tiles])  # [n_tiles, 3, H, W]
        pv_list.append(pv)
        np_list.append(pv.shape[0])

    return torch.cat(pv_list), np_list   # [sum_patches, 3, H, W]


# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║                      MODEL LOADING                                       ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def load_model():
    print(f"\n{'='*60}")
    print(f"  Model        : {MODEL_PATH}")
    print(f"  Quantization : 4-bit {QUANT_TYPE}  compute={QUANT_COMPUTE_DTYPE}")
    print(f"  Video source : {VIDEO_SOURCE}")
    print(f"  Questions    : {QUESTIONS_TO_ASK}")
    print(f"{'='*60}\n")

    # Do NOT pass torch_dtype or dtype — InternVideo2.5 custom __init__ rejects it.
    # Compute dtype handled by bnb_4bit_compute_dtype only.
    quant_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=QUANT_COMPUTE_DTYPE,
        bnb_4bit_use_double_quant=QUANT_DOUBLE_QUANT,
        bnb_4bit_quant_type=QUANT_TYPE,
    )

    model = AutoModel.from_pretrained(
        MODEL_PATH,
        trust_remote_code=True,
        device_map="auto",           # splits across both T4 GPUs
        quantization_config=quant_cfg,
    )
    model.eval()

    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
    print("Model loaded.\n")
    return model, tokenizer


# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║                      PROMPTS                                             ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

# ← PASTE YOUR NARRATIVE PROMPT HERE
NARRATIVE_PROMPT = """Task:
Carefully analyze the video frame-by-frame from the ego-car’s perspective and generate a clear, grounded description of the environment.
Focus on visual cues related to sky, weather, and lighting, and describe what you observe in natural language.
What to Observe and Describe-
Sky & Time-related cues
Is the sky visible or not
Sky color (blue, dark, orange, grey, etc.)
Sun visibility or absence
Does it feel like bright daytime, early morning (sun rising), evening (sun setting), or dark night
If sky is not visible, explicitly mention that
Weather-related cues
Whether the scene looks clear, cloudy, or mixed (that is its sunny but clouds are there)
Any signs of rain (raindrops on windshield, wipers moving, wet road reflections)
Fog or haze (reduced visibility, blurred distant objects)
Snow presence (on road or surroundings)
Road condition (dry, wet, reflective, snow-covered)
tell us about Lighting conditions
Also in cases where you see ego car going inside a tunnel do weather and daytime analysis before it enters tunnel
if no sky is visible like car is in tunnel then tell about lighting in tunnel
Whether scene is mainly lit by sunlight or headlight or streetlight
Output Format-
Describe video:
<description>
Sky/Time Observations:
<Description>
Weather Observations:
<Description>
Lighting Observations:
<Description>

"""


def build_classification_prompt(narrative: str, questions_to_ask: list) -> str:
    """
    Build Pass 2 prompt.
    Questions and options are formatted exactly as they appear in ALL_INCIDENT_QUESTIONS.
    """
    active = [q for q in ALL_INCIDENT_QUESTIONS if q[0] in questions_to_ask]

    lines = [
        "You are an expert traffic dashcam video environment and condition classifier.",
        "",
        "You have already watched this dashcam video and produced the following description:",
        "",
        "=== INCIDENT NARRATIVE ===",
        narrative,
        "",
        "=== YOUR TASK ===",
        "Using the incident narrative above AND the video frames,",
        "identify ONLY the environmental and road conditions present in the scene.",
        "Answer ONLY the questions listed below.",
        "For each question choose EXACTLY one option from the Options list provided.",
        "Your answer must be word-for-word one of the listed options — do not paraphrase.",
        "Respond with ONLY a valid JSON object.",
        "No markdown, no explanation, no text outside the JSON.",
        "Start with { and end with }.",
        "",
        "=== RULES ===",
        "1. Base your answers on clear visual evidence from the video.",
        "2. Do NOT infer conditions unless there is strong visual indication.",
        "3. If a condition is not clearly visible, select 'None' or 'Unknown' as appropriate.",
        "4. Wet road should ONLY be selected if reflections, splashes, or visible water are present.",
        "5. Fog/smoke must visibly reduce clarity or contrast.",
        "6. Snow/ice must be visible on road or surroundings.",
        "",
        "=== QUESTIONS ===",
        "",
    ]

    for key, desc, codes in active:
        # Format options exactly as they appear in ALL_INCIDENT_QUESTIONS
        opts = " | ".join(codes.keys())
        lines.append(f"[{key}] {desc}")
        lines.append(f"  Options: {opts}")
        lines.append("")

    json_template = "{\n" + ",\n".join(
        f'  "{k}": "<exact option text>"' for k in questions_to_ask
    ) + "\n}"
    lines.append(json_template)

    return "\n".join(lines)


# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║                      VLM INFERENCE                                       ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

@torch.no_grad()
def run_pass1_narrative(model, tokenizer,
                        pixel_values: torch.Tensor,
                        num_patches_list: list) -> tuple:
    """Pass 1: describe the incident. Returns (narrative_str, chat_history)."""
    video_prefix = "".join(
        [f"Frame{i+1}: <image>\n" for i in range(len(num_patches_list))]
    )
    question = video_prefix + NARRATIVE_PROMPT

    gen_cfg = dict(BASE_GENERATION_CONFIG)
    gen_cfg["max_new_tokens"] = MAX_NEW_TOKENS_NARRATIVE

    narrative, history = model.chat(
        tokenizer,
        pixel_values,
        question,
        gen_cfg,
        num_patches_list=num_patches_list,
        history=None,
        return_history=True,
    )
    return narrative, history


@torch.no_grad()
def run_pass2_classification(model, tokenizer,
                              pixel_values: torch.Tensor,
                              num_patches_list: list,
                              chat_history: list,
                              classification_prompt: str,
                              attempt: int = 0) -> str:
    """
    Pass 2: answer classification questions using narrative in history.
    return_history=False → model returns a plain string, NOT a tuple.
    """
    user_text = classification_prompt
    if attempt > 0:
        user_text += (
            "\n\nIMPORTANT: Previous response was not valid JSON. "
            "Output ONLY a raw JSON object. Start with { end with }. "
            "No markdown, no explanation outside the JSON."
        )

    gen_cfg = dict(BASE_GENERATION_CONFIG)
    gen_cfg["max_new_tokens"] = MAX_NEW_TOKENS_CLASSIFY

    # return_history=False → returns a plain str, do NOT unpack as tuple
    response = model.chat(
        tokenizer,
        pixel_values,
        user_text,
        gen_cfg,
        num_patches_list=num_patches_list,
        history=chat_history,
        return_history=False,
    )
    return response


def parse_json_response(raw: str) -> dict:
    text = re.sub(r"```(?:json)?\s*", "", raw.strip()).strip().rstrip("`").strip()

    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return {k: str(v).strip() for k, v in obj.items()}
    except (json.JSONDecodeError, ValueError):
        pass

    start, end = text.find("{"), text.rfind("}")
    if start != -1 and end > start:
        try:
            obj = json.loads(text[start:end + 1])
            if isinstance(obj, dict):
                return {k: str(v).strip() for k, v in obj.items()}
        except (json.JSONDecodeError, ValueError):
            pass

    result = {}
    for key in ALL_Q_KEYS:
        m = re.search(rf'"{re.escape(key)}"\s*:\s*"([^"]+)"', text)
        if m:
            result[key] = m.group(1).strip()
    return result


# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║                      POST-PROCESSING / VALIDATION                        ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def validate_codes(codes: dict) -> dict:
    out = {}
    for k in ALL_Q_KEYS:
        val = codes.get(k, UNMAPPABLE_CODE)
        try:
            val = int(val)
        except (ValueError, TypeError):
            val = UNMAPPABLE_CODE
        if val not in VALID_CODES.get(k, set()):
            val = UNMAPPABLE_CODE
        out[k] = val

    primary = out.get("Q5b", UNMAPPABLE_CODE)
    if primary in NON_VEHICLE_PRIMARY:
        out["Q5c"] = -2

    secondary = out.get("Q5f", UNMAPPABLE_CODE)
    if secondary in NON_VEHICLE_SECONDARY:
        out["Q5g"] = -2

    return out


# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║                      SINGLE VIDEO PIPELINE                               ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def process_single_video(video_id: int,
                         frames_dir: str,
                         model,
                         tokenizer,
                         verbose: bool = False) -> tuple:
    """
    Full two-pass pipeline for one video using pre-sampled JPEG frames.

    Args:
        video_id   : integer video ID (for logging)
        frames_dir : path to folder containing frame_*.jpg files
        model      : loaded InternVideo2.5 model
        tokenizer  : loaded tokenizer
        verbose    : print narrative and intermediate outputs if True

    Returns:
        codes     : {Q_KEY: int}
        raw_words : {Q_KEY: str}
        narrative : str
    """
    default_codes = {k: (UNMAPPABLE_CODE if k in QUESTIONS_TO_ASK else UNKNOWN_CODE)
                     for k in ALL_Q_KEYS}
    default_words = {k: ("Unknown" if k in QUESTIONS_TO_ASK else "Not asked")
                     for k in ALL_Q_KEYS}

    # ── Load frames ────────────────────────────────────────────────────────
    try:
        pixel_values, num_patches_list = load_sampled_frames(frames_dir)
        # Cast to model's dtype — model loads fp16 by default (no torch_dtype passed)
        pixel_values = pixel_values.to(model.dtype).to(model.device)
    except Exception as e:
        print(f"  [Video {video_id}] Frame load error: {e}")
        return default_codes, default_words, f"Frame load failed: {e}"

    if verbose:
        print(f"  Loaded {len(num_patches_list)} frames from {frames_dir}")

    # ── PASS 1: NARRATIVE ──────────────────────────────────────────────────
    narrative    = "Narrative generation failed."
    chat_history = []

    try:
        narrative, chat_history = run_pass1_narrative(
            model, tokenizer, pixel_values, num_patches_list
        )
        if verbose:
            print(f"\n  === PASS 1 NARRATIVE ===\n{narrative}\n")
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        gc.collect()
        print(f"  [Video {video_id}] OOM in narrative pass.")
        return default_codes, default_words, "OOM during narrative."
    except Exception as e:
        print(f"  [Video {video_id}] Narrative pass error: {e}")

    # ── PASS 2: CLASSIFICATION ─────────────────────────────────────────────
    classification_prompt = build_classification_prompt(narrative, QUESTIONS_TO_ASK)
    raw_words = {}

    for attempt in range(MAX_RETRIES + 1):
        try:
            raw_response = run_pass2_classification(
                model, tokenizer, pixel_values, num_patches_list,
                chat_history, classification_prompt, attempt=attempt,
            )
            if verbose:
                print(f"\n  === PASS 2 RESPONSE (attempt {attempt+1}) ===")
                print(raw_response)

            parsed = parse_json_response(raw_response)

            if verbose:
                print("\n  === PARSED JSON ===")
                for k, v in parsed.items():
                    print(f"    {k}: {v}")

            if len(parsed) >= max(1, len(QUESTIONS_TO_ASK) // 2):
                raw_words = {k: parsed.get(k, "Unknown") for k in QUESTIONS_TO_ASK}
                break

        except torch.cuda.OutOfMemoryError:
            print(f"  [Video {video_id}] OOM in classification attempt {attempt+1}.")
            torch.cuda.empty_cache()
            gc.collect()
            break
        except Exception as e:
            print(f"  [Video {video_id}] Classification error attempt {attempt+1}: {e}")

    # Merge into full result
    result_words = dict(default_words)
    for k in QUESTIONS_TO_ASK:
        result_words[k] = raw_words.get(k, "Unknown") if raw_words else "Unknown"

    codes_raw = {k: (word_to_code(result_words[k], k)
                     if k in QUESTIONS_TO_ASK else UNKNOWN_CODE)
                 for k in ALL_Q_KEYS}
    codes = validate_codes(codes_raw)

    if verbose:
        print("\n  === FINAL CODES ===")
        for k in ALL_Q_KEYS:
            tag = "" if k in QUESTIONS_TO_ASK else " [not asked]"
            print(f"    {k}: {codes[k]:>3}   raw='{result_words[k]}'{tag}")

    del pixel_values
    torch.cuda.empty_cache()

    return codes, result_words, narrative


# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║                      DEBUG — SINGLE VIDEO                                ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def debug_single_video(video_id: int, model=None, tokenizer=None):
    """
    Run the full pipeline on one video and print everything.

    Usage:
        model, tokenizer = load_model()
        debug_single_video(17, model, tokenizer)
        debug_single_video(660, model, tokenizer)   # reuse model
    """
    print(f"\n{'='*60}\n  DEBUG: Video {video_id}\n{'='*60}")

    frames_dir = frames_dir_for(video_id)
    if not os.path.isdir(frames_dir):
        print(f"ERROR: Frames dir not found: {frames_dir}")
        return None

    own_model = model is None
    if own_model:
        model, tokenizer = load_model()

    t0 = time.time()
    codes, raw_words, narrative = process_single_video(
        video_id, frames_dir, model, tokenizer, verbose=True
    )
    elapsed = time.time() - t0

    print(f"\n{'='*60}\n  RESULTS — Video {video_id}  ({elapsed:.1f}s)\n{'='*60}")
    print("\n  NARRATIVE:\n  " + "-"*38)
    print(narrative)
    print("\n  RAW WORDS → CODES:\n  " + "-"*38)
    for k in ALL_Q_KEYS:
        tag = "" if k in QUESTIONS_TO_ASK else " [not asked]"
        print(f"  {k:6s}  code={codes[k]:>3}   raw='{raw_words[k]}'{tag}")
    print(f"{'='*60}\n")

    if own_model:
        del model, tokenizer
        torch.cuda.empty_cache()
        gc.collect()

    return codes, raw_words, narrative


# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║                      SAVE SUBMISSIONS                                    ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def _get_output_columns() -> tuple:
    """
    Return (vid_col, q_cols) by matching QUESTIONS_TO_ASK keys against
    sample_submission.csv headers, or fall back to KEY_TO_COLUMN values.
    """
    asked_cols = [KEY_TO_COLUMN[k] for k in QUESTIONS_TO_ASK if k in KEY_TO_COLUMN]

    if os.path.exists(SAMPLE_SUB_PATH):
        df_h    = pd.read_csv(SAMPLE_SUB_PATH, nrows=0)
        all_h   = df_h.columns.tolist()
        vid_col = all_h[0]
        our_set = set(asked_cols)
        q_cols  = [c for c in all_h[1:] if c in our_set]
        if q_cols:
            return vid_col, q_cols

    return "video_id", asked_cols


def save_submissions(results_codes: dict,
                     results_words: dict,
                     words_path: str = SUBMISSION_WORDS_PATH,
                     codes_path: str = SUBMISSION_CODES_PATH):
    vid_col, q_cols = _get_output_columns()
    col_to_key      = {v: k for k, v in KEY_TO_COLUMN.items()}

    word_rows, code_rows = [], []
    for vid in sorted(results_codes.keys()):
        w_row = {vid_col: int(vid)}
        c_row = {vid_col: int(vid)}
        for col in q_cols:
            key        = col_to_key.get(col)
            w_row[col] = results_words[vid].get(key, "Unknown")       if key else "Unknown"
            c_row[col] = results_codes[vid].get(key, UNMAPPABLE_CODE) if key else UNMAPPABLE_CODE
        word_rows.append(w_row)
        code_rows.append(c_row)

    all_cols = [vid_col] + q_cols
    df_words = (pd.DataFrame(word_rows, columns=all_cols)
                if word_rows else pd.DataFrame(columns=all_cols))
    df_codes = (pd.DataFrame(code_rows, columns=all_cols)
                if code_rows else pd.DataFrame(columns=all_cols))

    df_words[vid_col] = df_words[vid_col].astype(int)
    df_codes[vid_col] = df_codes[vid_col].astype(int)

    df_words.to_csv(words_path, index=False)
    df_codes.to_csv(codes_path, index=False)

    print(f"  Words CSV : {words_path}  {df_words.shape}")
    print(f"  Codes CSV : {codes_path}  {df_codes.shape}")
    return df_words, df_codes


def save_narratives(results_narratives: dict,
                    path: str = NARRATIVES_JSON_PATH):
    # Convert int keys to str for JSON compatibility
    data = {str(vid): narr for vid, narr in results_narratives.items()}
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"  Narratives JSON : {path}  ({len(data)} entries)")


# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║                              MAIN                                        ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def discover_video_ids() -> list:
    """
    Scan SAMPLED_FRAMES_DIR for numeric folder names.
    Returns sorted list of (video_id, frames_dir) tuples in numeric order.
    Heatmap folders ({id}_heatmap) are ignored during discovery.
    """
    all_ids = []
    for entry in os.scandir(SAMPLED_FRAMES_DIR):
        if not entry.is_dir():
            continue
        name = entry.name
        # Skip heatmap folders during ID discovery
        if name.endswith("_heatmap"):
            continue
        try:
            vid = int(name)
            fdir = frames_dir_for(vid)   # resolves to correct source subfolder
            if os.path.isdir(fdir):
                all_ids.append((vid, fdir))
        except ValueError:
            continue

    all_ids.sort(key=lambda x: x[0])
    return all_ids


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    t0 = time.time()

    all_videos = discover_video_ids()
    print(f"Video IDs discovered : {len(all_videos)}")

    # First DATASET_PERCENT% in sorted numeric order
    n_use      = max(1, int(len(all_videos) * DATASET_PERCENT / 100.0))
    to_process = all_videos[:n_use]
    remaining  = all_videos[n_use:]

    print(f"Processing now       : {len(to_process)} ({DATASET_PERCENT}%)")
    print(f"Deferred             : {len(remaining)}\n")

    # Write unprocessed IDs immediately — survives a mid-run crash
    pd.DataFrame({"video_id": [v[0] for v in remaining]}).to_csv(
        UNPROCESSED_CSV_PATH, index=False
    )
    print(f"Unprocessed IDs saved: {UNPROCESSED_CSV_PATH}\n")

    model, tokenizer = load_model()

    results_codes, results_words, results_narratives, failures = {}, {}, {}, []

    for i, (vid, fdir) in enumerate(tqdm(to_process, desc="InternVideo2.5 VQA")):

        if VERBOSE and i % 5 == 0:
            elapsed = time.time() - t0
            eta     = (elapsed / max(i, 1)) * (len(to_process) - i)
            print(f"\n[{i+1}/{len(to_process)}] Video {vid} | "
                  f"{elapsed:.0f}s elapsed | ETA ~{eta:.0f}s")

        try:
            codes, words, narrative = process_single_video(
                vid, fdir, model, tokenizer, verbose=False
            )
            results_codes[vid] = codes
            results_words[vid] = words
            results_narratives[vid] = narrative  # ← collect narrative

            if VERBOSE:
                print("  " + "  ".join(f"{k}='{words.get(k, 'Unknown')}'" for k in QUESTIONS_TO_ASK))

        except Exception as e:
            print(f"  FAILED video {vid}: {e}")
            traceback.print_exc()
            results_codes[vid] = {k: UNMAPPABLE_CODE for k in ALL_Q_KEYS}
            results_words[vid] = {k: "Processing failed"       for k in ALL_Q_KEYS}
            results_narratives[vid] = "Processing failed"   #changes ← add this
            failures.append(vid)
            failures.append(vid)

        if (i + 1) % SAVE_INTERMEDIATE_EVERY == 0:
            print(f"\n  Checkpoint at video {i+1}...")
            save_submissions(results_codes, results_words)
            save_narratives(results_narratives) #changes

        gc.collect()

    print(f"\n{'='*60}\n  SAVING FINAL OUTPUTS\n{'='*60}")
    save_submissions(results_codes, results_words)
    save_narratives(results_narratives)   #changes

    elapsed = time.time() - t0
    print(f"\n{'='*60}")
    print(f"  DONE")
    print(f"  Processed  : {len(results_codes)} videos")
    print(f"  Failures   : {len(failures)}"
          f"{f'  {failures[:10]}' if failures else ''}")
    print(f"  Time       : {elapsed:.1f}s  "
          f"({elapsed / max(len(results_codes), 1):.1f}s/video)")
    print(f"{'='*60}")


# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║         RESUME CELL — copy into a new notebook cell                      ║
# ╚═══════════════════════════════════════════════════════════════════════════╝
"""
import os, gc, time, traceback
import pandas as pd
from tqdm.auto import tqdm

RESUME_UNPROCESSED_CSV = "/kaggle/input/YOUR_DATASET/unprocessed_video_ids.csv"
RESUME_PERCENT         = 60.0
RESUME_REMAINING_CSV   = os.path.join(OUTPUT_DIR, "unprocessed_video_ids_next.csv")

df_up        = pd.read_csv(RESUME_UNPROCESSED_CSV)
all_up       = df_up["video_id"].astype(int).tolist()
n_now        = max(1, int(len(all_up) * RESUME_PERCENT / 100.0))
ids_now      = all_up[:n_now]
ids_deferred = all_up[n_now:]

print(f"Unprocessed total : {len(all_up)}")
print(f"Processing now    : {n_now}")
print(f"Deferring         : {len(ids_deferred)}")
pd.DataFrame({"video_id": ids_deferred}).to_csv(RESUME_REMAINING_CSV, index=False)

video_list = []
for vid in ids_now:
    fdir = frames_dir_for(vid)
    if os.path.isdir(fdir):
        video_list.append((vid, fdir))
    else:
        print(f"  WARNING: {fdir} not found, skipping.")
print(f"Videos available  : {len(video_list)}")

model, tokenizer = load_model()

results_codes, results_words, failures = {}, {}, []
t0 = time.time()

for i, (vid, fdir) in enumerate(tqdm(video_list, desc="Resume VQA")):
    try:
        codes, words, _ = process_single_video(vid, fdir, model, tokenizer, verbose=False)
        results_codes[vid] = codes
        results_words[vid] = words
    except Exception as e:
        print(f"  FAILED {vid}: {e}")
        traceback.print_exc()
        results_codes[vid] = {k: UNMAPPABLE_CODE for k in ALL_Q_KEYS}
        results_words[vid] = {k: "Unknown"       for k in ALL_Q_KEYS}
        failures.append(vid)
    if (i + 1) % SAVE_INTERMEDIATE_EVERY == 0:
        print(f"  Checkpoint {i+1}...")
        save_submissions(results_codes, results_words)
    gc.collect()

save_submissions(results_codes, results_words)
elapsed = time.time() - t0
print(f"Done. {len(results_codes)} videos in {elapsed:.1f}s. Failures: {len(failures)}")
"""

# ── Entry point (comment out before running Resume Cell) ────────────────────
if __name__ == "__main__":
    if DEBUG_VIDEO_ID is not None:
        debug_single_video(DEBUG_VIDEO_ID)
    else:
        main()

Video IDs discovered : 661
Processing now       : 661 (100.0%)
Deferred             : 0

Unprocessed IDs saved: /kaggle/working/unprocessed_video_ids.csv


  Model        : OpenGVLab/InternVideo2_5_Chat_8B
  Quantization : 4-bit nf4  compute=torch.float16
  Video source : videos
  Questions    : ['Q1a', 'Q1b', 'Q2']



config.json: 0.00B [00:00, ?B/s]

configuration_internvl_chat.py: 0.00B [00:00, ?B/s]

configuration_intern_vit.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/OpenGVLab/InternVideo2_5_Chat_8B:
- configuration_intern_vit.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

special_tokens_map.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Model loaded.



InternVideo2.5 VQA:   0%|          | 0/661 [00:00<?, ?it/s]


[1/661] Video 0 | 156s elapsed | ETA ~102825s
  Q1a='Night'  Q1b='Sunny/Clear'  Q2='Streetlights on'
  Q1a='Day'  Q1b='Sunny/Clear'  Q2='Daylight'
  Q1a='Day'  Q1b='Sunny/Clear'  Q2='Daylight'
  Q1a='Day'  Q1b='Sunny/Clear'  Q2='Daylight'
  Q1a='Night'  Q1b='None'  Q2='Streetlights on'

[6/661] Video 5 | 259s elapsed | ETA ~34043s
  Q1a='Day'  Q1b='Cloudy'  Q2='Daylight'
  Q1a='Day'  Q1b='Sunny/Clear'  Q2='Daylight'
  Q1a='Day'  Q1b='Cloudy'  Q2='Daylight'
  Q1a='Day'  Q1b='Sunny/Clear'  Q2='Daylight'
  Q1a='Night'  Q1b='Rainy'  Q2='Streetlights on'

[11/661] Video 10 | 359s elapsed | ETA ~23382s
  Q1a='Night'  Q1b='Sunny/Clear'  Q2='Streetlights on'
  Q1a='Day'  Q1b='Cloudy'  Q2='Daylight'
  Q1a='Day'  Q1b='Sunny/Clear'  Q2='Daylight'
  Q1a='Day'  Q1b='Cloudy'  Q2='Daylight'
  Q1a='Day'  Q1b='Cloudy'  Q2='Daylight'

[16/661] Video 15 | 465s elapsed | ETA ~20008s
  Q1a='Night'  Q1b='Sunny/Clear'  Q2='Streetlights on'
  Q1a='Day'  Q1b='Sunny/Clear'  Q2='Daylight'
  Q1a='Dusk (sunset)' 